# 0824_lsw_004_imbalance_handling

검사유형별 5분리 구조(`0824_lsw_003_structure_comparison`에서 채택) 위에서 클래스 불균형 대응 기법을 비교합니다. 데이터 전처리와 평가 함수는 003과 동일하게 재사용합니다.

이번 노트북은 두 부분으로 구성됩니다: (1) SMOTE/ADASYN 적용 전에 소수 클래스(불량)에 이상치가 얼마나 있는지 가볍게 확인, (2) 불균형 처리 기법별 비교.

## 1. 설정과 라이브러리

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier

EXPERIMENT_ID = "0824_lsw_004_imbalance_handling"
RANDOM_STATE = 42
DATA_PATH = Path("../data/raw/dataset.csv")
TARGET = "class"
TIME_COLUMN = "timestamp"
RECORD_ID = "record_id"
MODEL_DIR = Path("../models")

COST_SCENARIOS = {"1:10": (1, 10), "1:100": (1, 100)}

assert DATA_PATH.exists(), f"파일을 찾을 수 없습니다: {DATA_PATH.resolve()}"
print("experiment:", EXPERIMENT_ID)

experiment: 0824_lsw_004_imbalance_handling


## 2. 데이터 로딩·전처리 (003과 동일)

In [2]:
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
source_index_column = raw_df.columns[0]
if source_index_column.startswith("Unnamed:"):
    raw_df = raw_df.rename(columns={source_index_column: RECORD_ID})
elif source_index_column != RECORD_ID:
    raise ValueError(f"예상하지 못한 첫 번째 컬럼: {source_index_column}")
assert raw_df[RECORD_ID].is_unique, "record_id가 고유하지 않습니다."

dedup_columns = [c for c in raw_df.columns if c not in {RECORD_ID, TIME_COLUMN}]
duplicate_mask = raw_df.duplicated(subset=dedup_columns, keep="first")
clean_df = raw_df.loc[~duplicate_mask].copy().reset_index(drop=True)

clean_df[TIME_COLUMN] = pd.to_datetime(clean_df[TIME_COLUMN], errors="raise", utc=True)
clean_df = clean_df.sort_values([TIME_COLUMN, RECORD_ID], kind="stable").reset_index(drop=True)

feature_columns_all = [c for c in clean_df.columns if c not in {RECORD_ID, TIME_COLUMN, TARGET}]

timestamps = clean_df[TIME_COLUMN]
timestamp_group_sizes = timestamps.value_counts(sort=False).sort_index()
cumulative_rows = timestamp_group_sizes.cumsum().to_numpy()
train_end_time = timestamp_group_sizes.index[int(np.searchsorted(cumulative_rows, len(clean_df) * 0.60, side="left"))]
valid_end_time = timestamp_group_sizes.index[int(np.searchsorted(cumulative_rows, len(clean_df) * 0.80, side="left"))]

train_mask = timestamps <= train_end_time
valid_mask = (timestamps > train_end_time) & (timestamps <= valid_end_time)
test_mask = timestamps > valid_end_time

print("rows_after_dedup:", len(clean_df))
pd.DataFrame(
    [
        {"split": name, "rows": int(mask.sum())}
        for name, mask in [("train", train_mask), ("validation", valid_mask), ("test", test_mask)]
    ]
).set_index("split")

rows_after_dedup: 391992


,rows
split,
train,235222
validation,78374
test,78396


## 3. 평가 함수 (003과 동일)

In [3]:
def slip_rate(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    actual_positive = y_true == 1
    if actual_positive.sum() == 0:
        return 0.0
    fn = ((y_pred == 0) & actual_positive).sum()
    return fn / actual_positive.sum()


def volume_reduction(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    actual_negative = y_true == 0
    if actual_negative.sum() == 0:
        return 0.0
    tn = ((y_pred == 0) & actual_negative).sum()
    return tn / actual_negative.sum()


def total_cost(y_true, y_pred, cost_fp, cost_fn):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    return fn * cost_fn + fp * cost_fp


def select_threshold(y_val, proba_val, max_slip_rate=0.01):
    candidates = np.sort(np.unique(proba_val))[::-1]
    for t in candidates:
        y_pred = (proba_val >= t).astype(int)
        if slip_rate(y_val, y_pred) <= max_slip_rate:
            return float(t)
    return 0.0


def evaluate_at_threshold(y_true, proba, threshold):
    y_pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    result = {
        "threshold": threshold,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "slip_rate": slip_rate(y_true, y_pred),
        "volume_reduction": volume_reduction(y_true, y_pred),
        "pr_auc": average_precision_score(y_true, proba),
        "roc_auc": roc_auc_score(y_true, proba) if len(np.unique(y_true)) > 1 else float("nan"),
    }
    for name, (cost_fp, cost_fn) in COST_SCENARIOS.items():
        result[f"total_cost_{name}"] = total_cost(y_true, y_pred, cost_fp, cost_fn)
    return result


def get_non_constant_columns(candidate_columns, train_frame):
    nunique = train_frame[candidate_columns].nunique()
    return nunique[nunique > 1].index.tolist()

## 4. 검사유형별 subset 준비

`0824_lsw_003`에서 채택한 구조(검사유형별 5분리, 유형별 임계값)를 이번 실험의 baseline으로 그대로 재현합니다.

In [4]:
type_splits = {}
for inspection_type in sorted(clean_df["inspection_type"].unique()):
    type_mask = clean_df["inspection_type"] == inspection_type
    type_train_df = clean_df.loc[train_mask & type_mask]
    type_valid_df = clean_df.loc[valid_mask & type_mask]
    type_test_df = clean_df.loc[test_mask & type_mask]
    type_feature_columns = get_non_constant_columns(feature_columns_all, type_train_df)
    type_splits[inspection_type] = {
        "train": type_train_df,
        "valid": type_valid_df,
        "test": type_test_df,
        "feature_columns": type_feature_columns,
    }
    print(f"type {inspection_type}: train={len(type_train_df)}, features={len(type_feature_columns)}, "
          f"train_pos={int((type_train_df[TARGET]==1).sum())}")

type 0: train=41961, features=47, train_pos=57
type 1: train=34041, features=34, train_pos=555


type 2: train=74910, features=24, train_pos=579


type 3: train=80792, features=22, train_pos=620
type 4: train=3518, features=24, train_pos=13


## 5. SMOTE/ADASYN 적용 전 — 소수 클래스(불량) 이상치 체크 (수정판)

**1차 시도의 결함**: 처음엔 정상+불량을 섞은 Train 전체로 IQR "정상 범위"를 구했는데, 불량 비율이 1~6%뿐이라 이 범위는 사실상 정상(다수) 클래스의 분포와 같다. 불량의 40~54%가 이 범위를 벗어난다는 결과는 "불량이 이상치"라는 뜻이 아니라 "불량과 정상의 분포가 원래 다르다(=오히려 좋은 신호)"는 뜻일 수 있다는 지적을 받았다.

**확인할 것 두 가지**:
1. 지금 기준(피처의 20% 이상이 범위 이탈)이 **정상 클래스에도 비슷하게 많이 걸리는지** — 그렇다면 기준 자체가 이 데이터에 비해 너무 헐거운 것이지, 불량이 특별히 이상한 게 아니다.
2. 임계값을 여러 단계로 바꿔가며 불량이 "뚝 떨어지는 절벽 구간"이 있는지 — 있다면 그 지점을 데이터 기반 임계값으로 쓴다.

방법을 두 가지 고친다: (a) IQR "정상 범위"는 **정상(class=0) 클래스만으로** 계산한다(불량이 섞여 범위가 넓어지는 것을 방지). (b) 고정된 20% 기준 하나만 보지 않고, 여러 임계값에서 정상 vs 불량의 flagged 비율을 나란히 표로 본다.

In [5]:
thresholds_to_check = [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]

outlier_scores_by_type = {}
cliff_rows = []

for inspection_type, split in type_splits.items():
    train_df = split["train"]
    feature_columns = split["feature_columns"]

    majority_df = train_df.loc[train_df[TARGET] == 0, feature_columns]
    minority_df = train_df.loc[train_df[TARGET] == 1, feature_columns]

    # 정상(class=0) 클래스만으로 "정상 운영 범위"를 정의
    q1 = majority_df.quantile(0.25)
    q3 = majority_df.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    def outlier_feature_fraction(df):
        is_outside = (df < lower) | (df > upper)
        return is_outside.mean(axis=1)

    majority_score = outlier_feature_fraction(majority_df)
    minority_score = outlier_feature_fraction(minority_df)
    outlier_scores_by_type[inspection_type] = {"majority": majority_score, "minority": minority_score}

    for th in thresholds_to_check:
        cliff_rows.append(
            {
                "inspection_type": inspection_type,
                "threshold": th,
                "정상_flagged_%": (majority_score > th).mean() * 100,
                "불량_flagged_%": (minority_score > th).mean() * 100,
            }
        )

cliff_df = pd.DataFrame(cliff_rows)
minority_pivot = cliff_df.pivot(index="threshold", columns="inspection_type", values="불량_flagged_%")
majority_pivot = cliff_df.pivot(index="threshold", columns="inspection_type", values="정상_flagged_%")

print("=== 정상(다수) 클래스: 임계값별 flagged 비율(%) — 기준 자체가 헐거운지 확인 ===")
display(majority_pivot.round(2))

print("\n=== 불량(소수) 클래스: 임계값별 flagged 비율(%) — 절벽 구간 확인 ===")
display(minority_pivot.round(2))

=== 정상(다수) 클래스: 임계값별 flagged 비율(%) — 기준 자체가 헐거운지 확인 ===


inspection_type,0,1,2,3,4
threshold,,,,,
0.00,76.67,95.65,76.16,76.03,44.11
0.05,65.54,75.00,54.38,45.05,27.56
0.10,58.33,45.19,35.58,29.31,17.66
0.15,41.84,27.78,19.27,16.38,11.81
0.20,33.63,21.51,12.62,7.20,9.59
0.25,23.85,13.73,2.89,1.68,4.19
0.30,11.23,6.82,0.27,0.40,2.51
0.40,4.35,0.00,0.00,0.00,0.83
0.50,0.36,0.00,0.00,0.00,0.03



=== 불량(소수) 클래스: 임계값별 flagged 비율(%) — 절벽 구간 확인 ===


inspection_type,0,1,2,3,4
threshold,,,,,
0.00,100.00,99.46,68.91,95.48,69.23
0.05,89.47,96.04,32.82,74.68,69.23
0.10,64.91,83.42,21.42,40.65,61.54
0.15,61.40,63.42,14.51,18.55,61.54
0.20,49.12,42.70,6.39,8.39,53.85
0.25,40.35,21.26,0.69,3.39,7.69
0.30,26.32,4.50,0.17,0.00,7.69
0.40,8.77,0.36,0.00,0.00,0.00
0.50,1.75,0.00,0.00,0.00,0.00


In [6]:
ratio_pivot = minority_pivot / majority_pivot.replace(0, np.nan)
print("=== 불량/정상 flagged 비율 (1.0 = 차이 없음, 클수록 불량이 진짜로 더 튄다는 뜻) ===")
ratio_pivot.round(2)

=== 불량/정상 flagged 비율 (1.0 = 차이 없음, 클수록 불량이 진짜로 더 튄다는 뜻) ===


inspection_type,0,1,2,3,4
threshold,,,,,
0.00,1.30,1.04,0.90,1.26,1.57
0.05,1.37,1.28,0.60,1.66,2.51
0.10,1.11,1.85,0.60,1.39,3.48
0.15,1.47,2.28,0.75,1.13,5.21
0.20,1.46,1.99,0.51,1.16,5.62
0.25,1.69,1.55,0.24,2.02,1.83
0.30,2.34,0.66,0.65,0.00,3.06
0.40,2.02,120.67,NaN,NaN,0.00
0.50,4.84,NaN,NaN,NaN,0.00


## 6. 결론 및 다음 단계 (1차 — 이상치 체크까지, 수정판)

### 1차 시도(섞인 IQR, 고정 20% 기준)가 과장됐음을 확인

정상(class=0) 클래스만으로 "정상 범위"를 다시 정의하고, 같은 20% 기준을 **정상 클래스에도** 적용해보니:

| inspection_type | 정상 flagged(th=0.2) | 불량 flagged(th=0.2) | 불량/정상 비율 |
|---|---:|---:|---:|
| 0 | 33.6% | 49.1% | 1.46 |
| 1 | 21.5% | 42.7% | 1.99 |
| 2 | 12.6% | 6.4% | 0.51 (불량이 오히려 덜 튐) |
| 3 | 7.2% | 8.4% | 1.16 |
| 4 | 9.6% | 53.9% | 5.62 (표본 13개, 노이즈 큼) |

**type0/1은 정상 클래스도 20~34%나 flagged된다** — 즉 이 기준 자체가 이 데이터에서 헐겁고, 1차 분석의 "49%/42.5%" 헤드라인 수치는 상당 부분 "불량이 이상해서"가 아니라 "기준이 느슨해서"였다.

### 절벽(cliff)은 뚜렷하지 않았지만, 임계값을 올리면 진짜 신호가 드러난다

정상/불량 flagged 곡선은 완만하게 감소해서 단일 절벽은 없었다. 대신 임계값별 **불량/정상 비율**을 보면:

- **type0**: 임계값을 올릴수록 비율이 1.1~1.7 → 2.3~4.8로 커진다. 즉 완만한 수준(th≈0.2)에서는 정상과 큰 차이가 없지만, **극단적인 수준(th≥0.3, 불량 26~40%만 해당)에서는 진짜로 불량이 더 튄다.** 진짜 이상치 후보는 49%가 아니라 26~40% 수준으로 좁혀야 한다.
- **type1**: 비율이 th=0.3에서 0.66으로 **1 밑으로 떨어진다** — 즉 극단적인 구간에서는 오히려 불량이 정상보다 덜 튄다. 42.5%라는 헤드라인은 실질적으로 이상치 신호가 아니라 대부분 기준의 느슨함이었다.
- **type2**: 모든 임계값에서 비율이 1 미만 — 불량이 정상보다 오히려 덜 극단적이다. 이상치 우려 없음.
- **type3**: 비율이 1.1~2.0 수준으로 약한 신호만 있다.
- **type4**: 표본이 13개뿐이라 결론을 내리기 어렵다.

### 다음 단계

- **불량 샘플을 대량(40~54%) 제거하는 건 근거가 부족하다** — 상당 부분 실제 신호(진짜 불량 패턴)를 지우는 것이 된다. 이 진단만으로 SMOTE 적용을 막을 이유는 없다.
- type0만 극단적인 임계값(th≥0.3, 실질 이상치 후보 26~40%)에서 유의미한 분리가 있으므로, type0에서 SMOTE가 잘 안 되면 이 서브셋을 살펴보는 정도로 남겨둔다.
- 나머지 유형(1/2/3/4)은 이상치 제거 없이 바로 SMOTE/ADASYN을 시도한다.
- 다음 셀부터 불균형 처리 기법(class_weight/scale_pos_weight, SMOTE, ADASYN, undersampling)을 유형별로 비교한다.
